# RAG System for SEC Filing Analysis

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) system for answering questions about Apple and Tesla's SEC 10-K filings.

## Setup
1. Clone the repository
2. Install dependencies
3. Download PDFs
4. Index documents
5. Answer questions

**Time to run**: ~15-20 minutes (first run with model downloads)

## Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/yourusername/naive_rag.git
%cd naive_rag

In [ ]:
# Install dependencies quietly
!pip install -q -r requirements.txt

In [ ]:
# Optional: extra HF runtime deps
!pip install -q accelerate safetensors bitsandbytes
!pip install -q gradio  # Optional: for interactive UI

## Step 2: Import Libraries and Setup

In [ ]:
import os
from dotenv import load_dotenv
import sys
import json
from pathlib import Path

# Load HF token from ./.env.txt (optional)
load_dotenv(os.path.join("./.env.txt"))

# Add project to path
sys.path.insert(0, '/kaggle/working/naive_rag')  # For Kaggle

from rag_system import RAGSystem, run_evaluation

print("✓ Libraries imported successfully")

## Step 3: Download PDF Documents

In [ ]:
# For Kaggle: Datasets are already available
# For Colab: Download from provided sources

# Check if PDFs exist
import os

documents = [
    "10-Q4-2024-As-Filed.pdf",
    "tsla-20231231-gen.pdf"
]

missing_docs = []
for doc in documents:
    if not os.path.exists(doc):
        missing_docs.append(doc)
    else:
        print(f"✓ {doc} found")

if missing_docs:
    print(f"\n⚠ Missing documents: {missing_docs}")
    print("\nPlease ensure PDFs are in the current directory")
    print("For Kaggle: Use Datasets feature to add the PDFs")
else:
    print("\n✓ All documents found!")

## Step 4: Initialize RAG System

In [ ]:
# Initialize the RAG system
print("Initializing RAG system...")

rag = RAGSystem(
    model_name="mistralai/Mistral-7B-Instruct-v0.2",
    embedding_model="all-MiniLM-L6-v2",
    chunk_size=500,
    chunk_overlap=50,
    use_reranker=True
)

print("✓ RAG system initialized")

## Step 5: Index Documents

In [ ]:
# Define documents
documents_to_index = [
    {
        "path": "10-Q4-2024-As-Filed.pdf",
        "name": "Apple 10-K"
    },
    {
        "path": "tsla-20231231-gen.pdf",
        "name": "Tesla 10-K"
    }
]

# Ingest and index documents
print("Starting document ingestion and indexing...\n")
rag.ingest_documents(documents_to_index)
print("\n✓ Documents indexed successfully")

## Step 6: Save Index for Future Use

In [ ]:
# Save the index
index_dir = "./rag_index"
rag.save_index(index_dir)
print(f"✓ Index saved to {index_dir}")

## Step 7: Test Single Query

In [ ]:
# Test with a single question
test_question = "What was Apple's total revenue for the fiscal year ended September 28, 2024?"

print(f"Question: {test_question}\n")

result = rag.answer_question(test_question)

print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")

## Step 8: Run Full Evaluation on All 13 Questions

In [ ]:
# Run the full evaluation
results = run_evaluation(rag)

## Step 9: Display Results Summary

In [ ]:
import pandas as pd

# Create a summary dataframe
summary_data = []
for result in results:
    summary_data.append({
        "Question ID": result['question_id'],
        "Answer Preview": result['answer'][:80] + "..." if len(result['answer']) > 80 else result['answer'],
        "Sources": ", ".join(result['sources']) if result['sources'] else "None"
    })

df = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("EVALUATION RESULTS SUMMARY")
print("="*100 + "\n")
print(df.to_string(index=False))
print("\n" + "="*100)

## Step 10: Save Results to JSON

In [ ]:
# Save results
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✓ Results saved to evaluation_results.json")

# Display first result as example
print("\nExample result format:")
print(json.dumps(results[0], indent=2))

## Interactive Query Interface (Optional)

In [ ]:
# Create an interactive function for testing
def ask_rag(question: str):
    """
    Ask the RAG system a question about Apple or Tesla 10-K filings.
    
    Args:
        question: Your question about the SEC filings
    
    Returns:
        Formatted answer with sources
    """
    result = rag.answer_question(question)
    
    print("\n" + "="*80)
    print("QUESTION:")
    print(question)
    print("\n" + "-"*80)
    print("ANSWER:")
    print(result['answer'])
    print("\n" + "-"*80)
    print("SOURCES:")
    if result['sources']:
        for source in result['sources']:
            print(f"  • {source}")
    else:
        print("  (No sources - out of scope or not found)")
    print("="*80 + "\n")

# Example usage
ask_rag("What types of vehicles does Tesla currently produce and deliver?")

## Try Your Own Questions!

Use the cell below to ask custom questions about Apple or Tesla's 10-K filings:

In [ ]:
# Try your own question here
my_question = "What was Tesla's total revenue for 2023?"  # Change this!
ask_rag(my_question)

## System Statistics and Metrics

In [ ]:
# Display system statistics
print("RAG SYSTEM STATISTICS")
print("="*50)
print("Total documents indexed: 2")
print(f"Total chunks created: {len(rag.vector_store.chunks)}")
print(f"Average chunk size: {sum(len(c['text']) for c in rag.vector_store.chunks) / len(rag.vector_store.chunks):.0f} chars")
print("Embedding model: all-MiniLM-L6-v2")
print("LLM model: microsoft/Phi-3-mini-4k-instruct")
print("Vector database: FAISS")
print("Re-ranking: Enabled (cross-encoder)")
print("="*50)

## Documentation

For more information, see:
- [README.md](https://github.com/yourusername/naive_rag/blob/main/README.md) - Full documentation
- [design.md](https://github.com/yourusername/naive_rag/blob/main/design.md) - Technical design report
- [GitHub Repository](https://github.com/yourusername/naive_rag) - Source code